#   LinkedIn Post Generator


In [8]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import base64
import re
import html
import json
import os
from datetime import datetime

# --- Global State ---
generated_post_content = ""
DRAFT_FILE = "linkedin_draft.json"
ANALYTICS_FILE = "post_analytics.json"

# --- Analytics Tracker ---
def load_analytics():
    if os.path.exists(ANALYTICS_FILE):
        with open(ANALYTICS_FILE, 'r') as f:
            return json.load(f)
    return {"total_posts": 0, "posts_by_type": {}, "posts_by_language": {}}

def save_analytics(post_type, language):
    analytics = load_analytics()
    analytics["total_posts"] = analytics.get("total_posts", 0) + 1
    analytics["posts_by_type"][post_type] = analytics["posts_by_type"].get(post_type, 0) + 1
    analytics["posts_by_language"][language] = analytics["posts_by_language"].get(language, 0) + 1
    with open(ANALYTICS_FILE, 'w') as f:
        json.dump(analytics, f, indent=2)

def show_analytics(b):
    analytics = load_analytics()
    with status_area:
        clear_output()
        print("="*50)
        print("📊 POST ANALYTICS DASHBOARD")
        print("="*50)
        print(f"📝 Total Posts Generated: {analytics.get('total_posts', 0)}")
        print("\n📌 By Post Type:")
        for ptype, count in analytics.get('posts_by_type', {}).items():
            bar = "█" * min(count, 20)
            print(f"   {ptype}: {count} {bar}")
        print("\n🌐 By Language:")
        for lang, count in analytics.get('posts_by_language', {}).items():
            bar = "█" * min(count, 20)
            print(f"   {lang}: {count} {bar}")
        print("="*50)

def reset_analytics(b):
    if os.path.exists(ANALYTICS_FILE):
        os.remove(ANALYTICS_FILE)
    with status_area:
        clear_output()
        print("✅ Analytics reset successfully!")

# =============================================================
# 1. SMART TEXT PROCESSING ENGINE
# =============================================================
def clean_text(text, make_title=False):
    text = text.strip()
    if not text:
        return ""
    if make_title:
        words = text.split()
        protected_words = []
        for w in words:
            if w.isupper() and len(w) > 1:
                protected_words.append(w)
            else:
                protected_words.append(w.capitalize())
        return " ".join(protected_words)
    sentences = re.split(r'(?<=[.!?])\s*', text)
    capitalized_sentences = [s[0].upper() + s[1:] for s in sentences if s.strip()]
    return " ".join(capitalized_sentences)


PROPER_NOUNS = {
    'ali', 'ahmed', 'hassan', 'fatima', 'sara', 'usman', 'zara', 'bilal', 'ayesha',
    'pakistan', 'karachi', 'lahore', 'islamabad', 'peshawar', 'quetta',
    'google', 'microsoft', 'amazon', 'meta', 'apple', 'netflix', 'openai',
    'python', 'pytorch', 'tensorflow', 'pandas', 'numpy', 'linkedin', 'github',
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
    'january', 'february', 'march', 'april', 'may', 'june',
    'july', 'august', 'september', 'october', 'november', 'december'
}

def capitalize_proper_nouns(text):
    def fix_word(w):
        clean = w.strip('.,!?:;()[]"\'')
        if clean.lower() in PROPER_NOUNS:
            return w.replace(clean, clean.capitalize(), 1)
        return w
    return ' '.join(fix_word(w) for w in text.split())


def get_article(word):
    if not word:
        return ""
    word_clean = word.strip().upper()
    vowel_sounds   = ['A', 'E', 'I', 'O', 'U']
    vowel_acronyms = ['L', 'M', 'N', 'R', 'S', 'X', 'H']
    if word_clean[0] in vowel_sounds or (len(word_clean) > 1 and word_clean[0] in vowel_acronyms):
        return "an"
    return "a"


def merge_and_format_tags(system_tags, user_tags_raw):
    final_tags_set = set(tag.strip() for tag in system_tags)
    if user_tags_raw.strip():
        raw_splits = re.split(r'[\s,]+', user_tags_raw)
        for tag in raw_splits:
            clean_tag = tag.replace('#', '').strip()
            if clean_tag:
                formatted_user_tag = "#" + clean_tag[0].upper() + clean_tag[1:]
                final_tags_set.add(formatted_user_tag)
    return " ".join(sorted(list(final_tags_set)))


# =============================================================
# 2. TEMPLATE BLOCKS WITH PROFESSIONAL ICONS
# =============================================================

# --- English blocks ---
def _en_recruiter_internship(title, details, acc, tech, tags):
    if not details.strip():
        details = "• Currently enrolled in BS/MSc (CS/IT/DS)\n• Strong foundation in Python/SQL\n• Eager to learn and contribute\n• Available for 3-6 months internship\n• Good communication and teamwork skills"
    return (
        f"🚨 New Opportunity Alert: '{title}'! 🚨\n\n"
        f"Hello network, our team is looking for talented interns. If you are passionate about "
        f"learning, driving impact, and working on modern systems, check out the parameters:\n\n"
        f"📌 Role Details & Requirements:\n{details}\n\n"
        f"Interested candidates can apply via link or drop CVs below. ➡️ Apply Now\n\n{tags}"
    )

def _en_recruiter_job(title, details, acc, tech, tags):
    if not details.strip():
        details = "• 2-4 years of relevant experience\n• Strong proficiency in Python/Data Science\n• Excellent problem-solving abilities\n• Bachelor's degree in CS/related field\n• Competitive salary + benefits package"
    return (
        f"🚨 New Career Opportunity: Openings for '{title}'! 🚨\n\n"
        f"Hello network, my organization is looking for technical professionals. "
        f"Check out the details:\n\n"
        f"📌 Core Requirements:\n{details}\n\n"
        f"Apply via the application portal. ➡️ Submit Application\n\n{tags}"
    )

def _en_user_internship(title, details, acc, tech, tags):
    if not details.strip():
        details = "• Working on real-world production systems\n• Learning from industry experts\n• Contributing to impactful projects\n• Building my professional network\n• Gaining hands-on experience with modern tech stack"
    art = get_article(title)
    return (
        f"🏆 Excited to share a personal milestone! I have accepted an offer and am starting "
        f"{art} '{title}' role!\n\n"
        f"I am incredibly grateful for this opportunity to step into the workspace and contribute "
        f"to real-world pipelines.\n\n"
        f"💼 What I'll be focusing on:\n{details}\n\n"
        f"A huge thank you to everyone who supported me along the way! 💪\n\n{tags}"
    )

def _en_user_job(title, details, acc, tech, tags):
    if not details.strip():
        details = "• Leading technical initiatives and projects\n• Collaborating with cross-functional teams\n• Architecting scalable solutions\n• Mentoring junior developers\n• Driving innovation and best practices"
    art = get_article(title)
    return (
        f"💼 Corporate Update: I'm thrilled to share that I'm starting a new position as "
        f"{art} '{title}'!\n\n"
        f"Looking forward to taking on new architectural challenges and collaborating with a brilliant team.\n\n"
        f"🚀 Core Responsibilities:\n{details}\n\n"
        f"Thank you to my network for the constant support! Let's build. ✨\n\n{tags}"
    )

def _en_project(title, details, acc, tech, tags):
    acc_line = f"• Global Accuracy Achieved: {acc}%\n" if acc else ""
    tech_line = tech if tech else "Modern Tech Stack"
    if not details.strip():
        details = "• End-to-end pipeline successfully deployed\n• Performance optimized and thoroughly tested\n• Complete documentation added\n• Ready for production use\n• Next phase planning has been initiated"
    return (
        f"🚀 Building in Public: Scaled my latest project '{title}'!\n\n"
        f"I'm excited to share that I have successfully completed and optimized the "
        f"end-to-end processing script.\n\n"
        f"📊 Core Details & Insights:\n"
        f"{acc_line}{details}\n\n"
        f"💻 Tech Stack: {tech_line}.\n\n{tags}"
    )

def _en_learning(title, details, acc, tech, tags):
    if not details.strip():
        details = "• Deep dive into core architecture patterns\n• Hands-on implementation of best practices\n• Documentation and knowledge sharing\n• Practical examples and use cases studied\n• Next steps and learning roadmap planned"
    return (
        f"📚 Today's Learning & Roadmap Share: '{title}'\n\n"
        f"Consistency is key in tech. Today, I focused on deep-diving into this domain "
        f"to strengthen my foundational grasp.\n\n"
        f"💡 Key Takeaways:\n{details}\n\n"
        f"What are you building today? Let's discuss in the comments! 💬\n\n{tags}"
    )

def _en_tips(title, details, acc, tech, tags):
    if not details.strip():
        details = "• Use list comprehensions for faster operations\n• Implement caching for repeated calculations\n• Profile your code before optimizing\n• Write modular and reusable functions\n• Always add proper error handling"
    return (
        f"💡 Quick Tech Tip: {title}\n\n"
        f"Here is a quick workflow optimization hack that can save you significant debugging sprints:\n\n"
        f"🔍 Implementation & Logic:\n{details}\n\n"
        f"Hope this adds value to your framework! Save it for later usage. 📌\n\n{tags}"
    )


# --- Roman Urdu blocks ---
def _ur_recruiter_internship(title, details, acc, tech, tags):
    if not details.strip():
        details = "• BS/MSc (CS/IT/DS) mein currently enrolled hon\n• Python/SQL mein strong foundation ho\n• Seekhne ka jazba aur contribution ho\n• 3-6 months internship ke liye available hon\n• Communication aur teamwork achi ho"
    return (
        f"🚨 Naya Internship Alert: '{title}'! 🚨\n\n"
        f"Hello tech community, hamari team ko aise interns ki talaash hai jo seekhne aur "
        f"real-world systems par kaam karne ka jazba rakhte hon.\n\n"
        f"📌 Requirements:\n{details}\n\n"
        f"Agar aap interested hain toh niche diye gaye link par apply karein. ➡️ Apply Karein\n\n{tags}"
    )

def _ur_recruiter_job(title, details, acc, tech, tags):
    if not details.strip():
        details = "• 2-4 saal ka relevant experience ho\n• Python/Data Science mein strong proficiency ho\n• Problem-solving skills behtareen hon\n• CS ya related field mein Bachelor's degree ho\n• Competitive salary + benefits package"
    return (
        f"🚨 Career ka Naya Mauka: Openings for '{title}'! 🚨\n\n"
        f"Hello network, hamari organization mein ek behtareen role available hai. "
        f"Details niche check karein:\n\n"
        f"📌 Requirements:\n{details}\n\n"
        f"Interested log portal par apply kar sakte hain. ➡️ Apply Karein\n\n{tags}"
    )

def _ur_user_internship(title, details, acc, tech, tags):
    if not details.strip():
        details = "• Real-world production systems par kaam karunga/karungi\n• Industry experts se seekhne ka mauka milega\n• Impactful projects mein contribution karunga/karungi\n• Apna professional network build karunga/karungi\n• Modern tech stack par hands-on experience"
    return (
        f"🏆 Nayi shuruat! Mujhe share karte hue behad khushi ho rahi hai ke maine offer "
        f"accept kar li hai aur main baqaida '{title}' ke taur par start kar raha/rahi hoon!\n\n"
        f"Main is mauke ke liye bohot shukargzar hoon jahan mujhe industrial level ka exposure milega.\n\n"
        f"💼 Mera main focus kya hoga:\n{details}\n\n"
        f"Un sab logon ka shukriya jinhone is safar mein mera sath diya! 💪\n\n{tags}"
    )

def _ur_user_job(title, details, acc, tech, tags):
    if not details.strip():
        details = "• Technical initiatives aur projects ki leadership karunga/karungi\n• Cross-functional teams ke saath collaboration\n• Scalable solutions ka architecture design\n• Junior developers ko mentorship\n• Innovation aur best practices ko drive karna"
    return (
        f"💼 Career Update: Mujhe yeh batate hue khushi ho rahi hai ke main ab naye position "
        f"'{title}' par apni nayi journey shuru kar raha/rahi hoon!\n\n"
        f"Naye challenges aur engineering optimizations ke liye fully excited hoon.\n\n"
        f"🚀 Mera Role aur Kaam:\n{details}\n\n"
        f"Mera sath karne ke liye pure network ka dil se shukriya! ✨\n\n{tags}"
    )

def _ur_project(title, details, acc, tech, tags):
    acc_line = f"• Hasil Kardah Accuracy: {acc}%\n" if acc else ""
    tech_line = tech if tech else "Python aur modern tools"
    if not details.strip():
        details = "• End-to-end pipeline successfully deploy ho chuki hai\n• Performance optimize aur test ho chuki hai\n• Complete documentation add kar di gayi hai\n• Production use ke liye ready hai\n• Next phase ki planning start kar di hai"
    return (
        f"🚀 Building in Public: Maine apna naya project '{title}' complete aur deploy kar liya hai!\n\n"
        f"Is architecture ki core processing ab flawlessly deploy ho chuki hai.\n\n"
        f"📊 Project ki Insights:\n"
        f"{acc_line}{details}\n\n"
        f"💻 Tech Stack: {tech_line}.\n\n{tags}"
    )

def _ur_learning(title, details, acc, tech, tags):
    if not details.strip():
        details = "• Core architecture patterns ko samjha\n• Best practices ko implement kiya\n• Documentation aur knowledge sharing ki\n• Practical examples aur use cases study kiye\n• Next steps aur learning roadmap plan kar liya"
    return (
        f"📚 Aaj ki Learning aur Roadmap Update: '{title}'\n\n"
        f"Tech mein consistency hi sab kuch hai. Aaj maine is domain ke internal structure "
        f"ko deep-dive kiya.\n\n"
        f"💡 Asal Takeaways:\n{details}\n\n"
        f"Aap aaj kal kya seekh rahe hain? Niche comments mein discuss karte hain! 💬\n\n{tags}"
    )

def _ur_tips(title, details, acc, tech, tags):
    if not details.strip():
        details = "• List comprehensions use karo faster operations ke liye\n• Repeated calculations ke liye caching implement karo\n• Code optimize karne se pehle profile karo\n• Modular aur reusable functions likho\n• Proper error handling add karo"
    return (
        f"💡 Aik Choti Tech Tip: {title}\n\n"
        f"Yeh ek aisa quick workflow hack hai jo aapka development ka kafi time bacha sakta hai:\n\n"
        f"🔍 Kaise use karein aur Logic:\n{details}\n\n"
        f"Umeed hai yeh aapke daily sprint mein kaam aayega! Isay save karlein. 📌\n\n{tags}"
    )


POST_TEMPLATES = {
    "English (Professional)": {
        "Recruiter - Internship Opportunity": _en_recruiter_internship,
        "Recruiter - Job Opportunity":        _en_recruiter_job,
        "User - Internship Offer":            _en_user_internship,
        "User - Job Offer":                   _en_user_job,
        "Project Upload":                     _en_project,
        "Daily Learning / Roadmap":           _en_learning,
        "Tips & Tricks":                      _en_tips,
    },
    "Roman Urdu (Conversational)": {
        "Recruiter - Internship Opportunity": _ur_recruiter_internship,
        "Recruiter - Job Opportunity":        _ur_recruiter_job,
        "User - Internship Offer":            _ur_user_internship,
        "User - Job Offer":                   _ur_user_job,
        "Project Upload":                     _ur_project,
        "Daily Learning / Roadmap":           _ur_learning,
        "Tips & Tricks":                      _ur_tips,
    }
}

BASE_TAGS_MAP = {
    "Recruiter - Internship Opportunity": ["#Hiring", "#InternshipOpportunity", "#TechInterns", "#Recruitment", "#DataScience"],
    "Recruiter - Job Opportunity":        ["#Hiring", "#JobOpportunity", "#TechJobs", "#CareerOpenings", "#Recruitment"],
    "User - Internship Offer":            ["#Internship", "#CareerUpdate", "#NewJourney", "#GrowthMindset"],
    "User - Job Offer":                   ["#NewJob", "#CareerGrowth", "#TechIndustry", "#Engineering", "#Jobs"],
    "Project Upload":                     ["#MachineLearning", "#DataScience", "#BuildingInPublic", "#TechInnovation"],
    "Daily Learning / Roadmap":           ["#ContinuousLearning", "#TechRoadmap", "#Python", "#TechCommunity"],
    "Tips & Tricks":                      ["#TechTips", "#PythonHacks", "#CodingLife", "#Productivity"],
}


# =============================================================
# 3. HYBRID INTENT CLASSIFIER
# =============================================================
def determine_final_intent(selected_dropdown, title, details):
    combined_text = f"{title.lower()} {details.lower()}"
    recruiter_triggers = ['hiring', 'apply now', 'opening', 'vacancy', 'opportunities',
                          'looking for', 'deadline', 'we are hiring', 'join our team']

    if selected_dropdown == "Internship Offer":
        if any(kw in combined_text for kw in recruiter_triggers):
            return "Recruiter - Internship Opportunity"
        return "User - Internship Offer"
    elif selected_dropdown == "Job Offer":
        if any(kw in combined_text for kw in recruiter_triggers):
            return "Recruiter - Job Opportunity"
        return "User - Job Offer"
    elif selected_dropdown == "Hiring / Opportunity Share":
        if any(kw in combined_text for kw in ['intern', 'internship']):
            return "Recruiter - Internship Opportunity"
        return "Recruiter - Job Opportunity"
    elif selected_dropdown == "Project Upload":
        if any(kw in combined_text for kw in ['tip', 'trick', 'hack', 'shortcut']):
            return "Tips & Tricks"
        return "Project Upload"
    return selected_dropdown


# =============================================================
# 4. IMPROVED FUNCTIONS
# =============================================================

def update_char_count(change):
    text = details_input.value
    count = len(text)
    char_label.value = f"📝 {count} / 3000 characters"
    if count > 3000:
        char_label.style.text_color = 'red'
    elif count > 2800:
        char_label.style.text_color = 'orange'
    else:
        char_label.style.text_color = 'green'

def sanitize_html(text):
    text = html.escape(text)
    text = re.sub(r'<[^>]*>', '', text)
    return text

def auto_save_draft():
    draft = {
        'title': title_input.value,
        'details': details_input.value,
        'tech': tech_input.value,
        'tags': tags_input.value,
        'language': language_dropdown.value,
        'category': category_dropdown.value,
        'accuracy': accuracy_input.value,
        'last_saved': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    with open(DRAFT_FILE, 'w') as f:
        json.dump(draft, f)

def manual_save_draft(b):
    auto_save_draft()
    with status_area:
        clear_output()
        print(f"💾 Draft saved at {datetime.now().strftime('%H:%M:%S')}")

def load_last_draft(b):
    if os.path.exists(DRAFT_FILE):
        with open(DRAFT_FILE, 'r') as f:
            draft = json.load(f)
        title_input.value = draft.get('title', '')
        details_input.value = draft.get('details', '')
        tech_input.value = draft.get('tech', '')
        tags_input.value = draft.get('tags', '')
        language_dropdown.value = draft.get('language', 'English (Professional)')
        category_dropdown.value = draft.get('category', 'Project Upload')
        accuracy_input.value = draft.get('accuracy', '')
        with status_area:
            clear_output()
            print(f"✅ Draft loaded! Last saved: {draft.get('last_saved', 'Unknown')}")
        update_char_count(None)
    else:
        with status_area:
            clear_output()
            print("⚠️ No draft found.")

def export_to_file(b):
    if generated_post_content:
        filename = f"linkedin_post_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(generated_post_content)
        with status_area:
            clear_output()
            print(f"📄 Exported to '{filename}'")
    else:
        with status_area:
            clear_output()
            print("⚠️ No post to export.")


# =============================================================
# 5. GENERATE POST
# =============================================================

loading_spinner = widgets.HTML(value="<div style='display:none; color:#0a66c2;'>⏳ Generating...</div>")

def generate_post(b):
    global generated_post_content
    
    loading_spinner.layout.display = 'block'
    generate_btn.disabled = True
    generate_btn.description = "⏳ Generating..."
    
    with output_area:
        clear_output()
        
        try:
            selected_lang = language_dropdown.value
            dropdown_val  = category_dropdown.value
            title         = clean_text(title_input.value, make_title=True)
            tech_stack    = clean_text(tech_input.value,  make_title=True)
            
            raw_details = details_input.value
            sanitized_details = sanitize_html(raw_details)
            details = capitalize_proper_nouns(clean_text(sanitized_details, make_title=False))
            custom_tags = sanitize_html(tags_input.value.strip())
            
            acc_raw = accuracy_input.value.strip()
            acc_filtered = re.sub(r'[^0-9.%]', '', acc_raw)
            acc_clean = acc_filtered.rstrip('%').strip() if acc_filtered else ""
            
            if not title and not raw_details:
                print("❌ Error: Please fill in Title or Details first!")
                generate_btn.disabled = False
                generate_btn.description = "⚡ Generate Post"
                loading_spinner.layout.display = 'none'
                return
            
            if len(raw_details) > 3000:
                print(f"⚠️ Warning: {len(raw_details)} chars (LinkedIn limit: 3000)")
            
            final_intent = determine_final_intent(dropdown_val, title, raw_details)
            merged_tags = merge_and_format_tags(BASE_TAGS_MAP[final_intent], custom_tags)
            
            generated_post_content = POST_TEMPLATES[selected_lang][final_intent](
                title, raw_details, acc_clean, tech_stack, merged_tags
            )
            
            save_analytics(final_intent, selected_lang)
            auto_save_draft()
            
            print("\n" + "="*60)
            print("🔵 LINKEDIN POST PREVIEW")
            print("="*60 + "\n")
            
            if file_upload.value:
                files_list = []
                if isinstance(file_upload.value, dict):
                    for f_name, f_info in file_upload.value.items():
                        files_list.append({'name': f_name, 'content': f_info['content']})
                elif isinstance(file_upload.value, (tuple, list)):
                    files_list = list(file_upload.value)
                
                image_files = [f for f in files_list if f['name'].lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.webp'))]
                other_files = [f for f in files_list if f not in image_files]
                
                if image_files:
                    print("📸 IMAGES ATTACHED:\n")
                    img_widgets = []
                    for img_item in image_files:
                        img_w = widgets.Image(
                            value=img_item['content'], 
                            format='png', 
                            width=220,
                            layout=widgets.Layout(
                                margin='5px', 
                                border='2px solid #0a66c2',
                                border_radius='10px'
                            )
                        )
                        img_widgets.append(img_w)
                    display(widgets.HBox(img_widgets, layout=widgets.Layout(flex_flow='row wrap')))
                    print("\n" + "─"*50 + "\n")
                
                for file_item in other_files:
                    f_name = file_item['name']
                    b64 = base64.b64encode(file_item['content']).decode()
                    display(HTML(f'<a href="data:application/octet-stream;base64,{b64}" download="{html.escape(f_name)}" style="color:#0a66c2;">📎 {html.escape(f_name)}</a>'))
                if other_files:
                    print("\n" + "─"*50 + "\n")
            
            display(HTML(f"""
            <div style="
                border: 1px solid #e0e0e0;
                border-radius: 16px;
                padding: 20px;
                margin: 15px 0;
                background: white;
                font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
                max-width: 600px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.05);
            ">
                <div style="display: flex; align-items: center; margin-bottom: 16px;">
                    <div style="
                        width: 48px; height: 48px;
                        background: #0a66c2;
                        border-radius: 50%;
                        display: flex;
                        align-items: center;
                        justify-content: center;
                        color: white;
                        font-weight: bold;
                        font-size: 18px;
                        margin-right: 12px;
                    ">👤</div>
                    <div>
                        <div style="font-weight: 700; color: #000; font-size: 15px;">Your Name</div>
                        <div style="font-size: 12px; color: #666;">Your Title • {datetime.now().strftime('%b %d')}</div>
                    </div>
                </div>
                <div style="
                    white-space: pre-wrap;
                    line-height: 1.5;
                    color: #333;
                    font-size: 14px;
                ">
                    {html.escape(generated_post_content).replace(chr(10), '<br>')}
                </div>
                <div style="
                    margin-top: 16px;
                    padding-top: 12px;
                    border-top: 1px solid #e9e9e9;
                    color: #666;
                    font-size: 12px;
                    display: flex;
                    gap: 20px;
                ">
                    <span>👍 Like</span> <span>💬 Comment</span> <span>🔄 Repost</span> <span>📤 Send</span>
                </div>
            </div>
            """))
            
            print("\n" + "="*60)
            print("📋 TEXT VERSION (Ready to Copy)")
            print("="*60 + "\n")
            print(generated_post_content)
            print(f"\n📊 Stats: {len(generated_post_content)} chars | Intent: {final_intent}")
            print("\n" + "="*60)
            
            display(action_button_layout, preview_export_layout)
            
        except Exception as e:
            print(f"❌ Error: {str(e)}")
        
        finally:
            loading_spinner.layout.display = 'none'
            generate_btn.disabled = False
            generate_btn.description = "⚡ Generate Post"


# =============================================================
# 6. UTILITY ACTIONS
# =============================================================

def copy_to_clipboard(b):
    escaped = html.escape(generated_post_content).replace('\n', '\\n').replace("'", "\\'")
    display(HTML(f"""
    <script>
    navigator.clipboard.writeText('{escaped}').then(() => alert('📋 Copied!')).catch(err => alert('Failed: '+err));
    </script>
    """))

def redirect_to_linkedin(b):
    display(HTML("<script>window.open('https://www.linkedin.com/feed/', '_blank');</script>"))

def on_file_upload(change):
    with status_area:
        clear_output()
        if change['new']:
            files = change['new']
            names = list(files.keys()) if isinstance(files, dict) else [f['name'] for f in files]
            print(f"✅ Loaded: {', '.join(names)}")

def on_category_change(change):
    c = change['new']
    if c == "Project Upload":
        accuracy_input.layout.display = 'flex'
        tech_input.layout.display = 'flex'
    elif c in ["Internship Offer", "Job Offer", "Hiring / Opportunity Share"]:
        accuracy_input.layout.display = 'none'
        tech_input.layout.display = 'none'
    else:
        accuracy_input.layout.display = 'none'
        tech_input.layout.display = 'flex'


# =============================================================
# 7. UI LAYOUT
# =============================================================

language_dropdown = widgets.Dropdown(
    options=['English (Professional)', 'Roman Urdu (Conversational)'],
    value='English (Professional)',
    description='🌐 Language:',
    style={'description_width': 'initial'}
)

category_dropdown = widgets.Dropdown(
    options=['Project Upload', 'Internship Offer', 'Job Offer',
             'Hiring / Opportunity Share', 'Daily Learning / Roadmap', 'Tips & Tricks'],
    value='Project Upload',
    description='📌 Post Type:',
    style={'description_width': 'initial'}
)

title_input = widgets.Text(
    value='', placeholder='e.g., Data Scientist, L-shaped Curve Analysis',
    description='📝 Title/Role:',
    style={'description_width': 'initial'}
)

tech_input = widgets.Text(
    value='', placeholder='e.g., Python, Pandas, PyTorch',
    description='💻 Tech Stack:',
    style={'description_width': 'initial'}
)

accuracy_input = widgets.Text(
    value='', placeholder='e.g., 94.2',
    description='🎯 Accuracy (%):',
    style={'description_width': 'initial'}
)

tags_input = widgets.Text(
    value='',
    placeholder='e.g., career, software, openSource',
    description='🏷️ Custom Tags:',
    style={'description_width': 'initial'}
)

char_label = widgets.HTML(value="📝 0 / 3000 characters", layout=widgets.Layout(margin='0 0 5px 100px'))

details_input = widgets.Textarea(
    value='',
    placeholder='Post details here... (Leave empty for professional default content)',
    description='📄 Context:',
    rows=8,
    layout=widgets.Layout(width='95%', height='auto')
)
details_input.observe(update_char_count, names='value')

file_upload = widgets.FileUpload(
    accept='', multiple=True,
    description='📎 Attach Files', button_style='info', icon='paperclip'
)
file_upload.observe(on_file_upload, names='value')

draft_save_btn = widgets.Button(description="💾 Save Draft", button_style='info', icon='save')
draft_load_btn = widgets.Button(description="📂 Load Draft", button_style='info', icon='folder-open')
draft_save_btn.on_click(manual_save_draft)
draft_load_btn.on_click(load_last_draft)
draft_layout = widgets.HBox([draft_save_btn, draft_load_btn])

analytics_btn = widgets.Button(description="📊 Show Analytics", button_style='info', icon='chart')
reset_analytics_btn = widgets.Button(description="🔄 Reset Analytics", button_style='danger', icon='trash')
analytics_btn.on_click(show_analytics)
reset_analytics_btn.on_click(reset_analytics)
analytics_layout = widgets.HBox([analytics_btn, reset_analytics_btn])

generate_btn = widgets.Button(
    description="⚡ Generate Post",
    button_style='success', icon='bolt',
    layout=widgets.Layout(width='95%', margin='10px 0px')
)
generate_btn.on_click(generate_post)

btn_copy = widgets.Button(description="📋 Copy Post", button_style='primary')
btn_linkedin = widgets.Button(description="🌐 Open LinkedIn", button_style='warning')
btn_copy.on_click(copy_to_clipboard)
btn_linkedin.on_click(redirect_to_linkedin)
action_button_layout = widgets.HBox([btn_copy, btn_linkedin])

preview_btn = widgets.Button(description="👁️ Preview Post", button_style='info')
export_btn = widgets.Button(description="📄 Export to File", button_style='info')
preview_btn.on_click(show_preview)
export_btn.on_click(export_to_file)
preview_export_layout = widgets.HBox([preview_btn, export_btn])

status_area = widgets.Output()
output_area = widgets.Output()
preview_area = widgets.Output()

category_dropdown.observe(on_category_change, names='value')


# =============================================================
# 8. RENDER UI
# =============================================================

def show_preview(b):
    if generated_post_content:
        preview_area.clear_output()
        with preview_area:
            display(HTML(f"""
            <div style="border: 2px solid #0a66c2; border-radius: 12px; padding: 20px; margin: 10px 0; background: #f8fafc;">
                <div style="display: flex; justify-content: space-between; margin-bottom: 10px;">
                    <strong style="color: #0a66c2;">👁️ Post Preview</strong>
                    <span style="color: gray; font-size: 12px;">{len(generated_post_content)} characters</span>
                </div>
                <div style="border-top: 1px solid #ddd; padding-top: 10px; white-space: pre-wrap; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;">
                    {html.escape(generated_post_content).replace(chr(10), '<br>')}
                </div>
            </div>
            """))
    else:
        preview_area.clear_output()
        print("⚠️ No post generated yet. Click 'Generate Post' first.")

print("="*30)
print("👑 LinkedIn Post Generator")
print("="*30)
display(
    language_dropdown,
    category_dropdown,
    title_input,
    tech_input,
    accuracy_input,
    details_input,
    char_label,
    tags_input,
    file_upload,
    draft_layout,
    analytics_layout,
    generate_btn,
    loading_spinner,
    status_area,
    preview_area,
    output_area
)

print("\n💡 Pro Tips:")
print("   • Leave 'Context' empty for professional default content")
print("   • Uploaded images appear ABOVE your post text")
print("   • Check Analytics to track your posting activity")
print("   • Use ➡️ instead of 👇 for professional CTA\n")

👑 LinkedIn Post Generator


Dropdown(description='🌐 Language:', options=('English (Professional)', 'Roman Urdu (Conversational)'), style=D…

Dropdown(description='📌 Post Type:', options=('Project Upload', 'Internship Offer', 'Job Offer', 'Hiring / Opp…

Text(value='', description='📝 Title/Role:', placeholder='e.g., Data Scientist, L-shaped Curve Analysis', style…

Text(value='', description='💻 Tech Stack:', placeholder='e.g., Python, Pandas, PyTorch', style=TextStyle(descr…

...

HTML(value="<div style='display:none; color:#0a66c2;'>⏳ Generating...</div>")

Output()

Output()

Output()


💡 Pro Tips:
   • Leave 'Context' empty for professional default content
   • Uploaded images appear ABOVE your post text
   • Check Analytics to track your posting activity
   • Use ➡️ instead of 👇 for professional CTA

